# 02. Credit Model Development & Champion-Challenger Benchmarking
**Project**: AI Model Risk & GenAI Evaluation Framework
**Focus**: Baseline (Logistic Regression) vs Champion (XGBoost) modeling with Fair Lending separation


In [ ]:
import os
import sys
import pandas as pd
import numpy as np

sys.path.insert(0, os.path.abspath('..'))
from src.model_training import CreditModelTrainer
from src.performance import PerformanceEvaluator


## 1. Prepare Data & Train Pipelines


In [ ]:
trainer = CreditModelTrainer(data_path='../data/processed/credit_risk_clean.csv')
splits = trainer.prepare_data(test_size=0.25, random_state=42)
models = trainer.train_models(splits)
trainer.save_artifacts(models_dir='../models', data_splits=splits)


## 2. Evaluate Baseline vs Champion Discrimination


In [ ]:
X_test = splits['X_test']
y_test = splits['y_test']

base_probs = models['baseline_logistic'].predict_proba(X_test)[:, 1]
champ_probs = models['champion_xgboost'].predict_proba(X_test)[:, 1]

evaluator = PerformanceEvaluator(default_threshold=0.5)
comparison_df = evaluator.compare_models(y_test, {
    'Baseline (Logistic Regression)': base_probs,
    'Champion (XGBoost)': champ_probs
})
comparison_df


## 3. Discrimination ROC and Precision-Recall Curves


In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, precision_recall_curve, auc

fpr_b, tpr_b, _ = roc_curve(y_test, base_probs)
fpr_c, tpr_c, _ = roc_curve(y_test, champ_probs)

plt.figure(figsize=(8, 6))
plt.plot(fpr_c, tpr_c, label=f'Champion XGBoost (AUC = {auc(fpr_c, tpr_c):.3f})', color='#002663', lw=2)
plt.plot(fpr_b, tpr_b, label=f'Baseline Logistic (AUC = {auc(fpr_b, tpr_b):.3f})', color='#f59e0b', lw=2, linestyle='--')
plt.plot([0, 1], [0, 1], 'k--', alpha=0.5)
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Discrimination Curves')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()
